In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import scipy.io as sio
from collections import defaultdict
from shutil import rmtree
from kilosort.io import load_ops
from kilosort.data_tools import (
    mean_waveform, cluster_templates, get_best_channel
)

# ---------------------- waveform analysis ----------------------

def analyze_waveform(waveform, fs):
    peak_idx = int(np.argmax(waveform))
    if peak_idx < waveform.size - 1:
        trough_idx = peak_idx + int(np.argmin(waveform[peak_idx:]))
    else:
        trough_idx = int(np.argmin(waveform))
    dt_us = (trough_idx - peak_idx) / fs * 1e6
    return {
        "peak_idx": peak_idx,
        "trough_idx": trough_idx,
        "peak_uV": float(waveform[peak_idx]),
        "trough_uV": float(waveform[trough_idx]),
        "peak_time_us": peak_idx / fs * 1e6,
        "trough_time_us": trough_idx / fs * 1e6,
        "peak_to_trough_us": dt_us
    }

# ---------------------- merged figure (waveform + ISI) ----------------------

def waveform_isi_fig(
    t_us, mean_wv_uV, mean_temp_uV, fs, cluster_id, session_name,
    best_chan, cluster_spikes, all_spikes, save_path_png
):

    # Waveform metrics
    wf = analyze_waveform(mean_wv_uV, fs)
    peak_idx, trough_idx, dt_us = wf["peak_idx"], wf["trough_idx"], wf["peak_to_trough_us"]

    # Avg firing rate
    n_spikes = int(cluster_spikes.size)
    rec_duration_s = all_spikes.max() / fs if all_spikes.size > 0 else np.nan
    fr_hz = n_spikes / rec_duration_s if rec_duration_s else float('nan')

    # ISI histogram
    isi_us = np.diff(cluster_spikes) / fs * 1e6
    bins_us = np.arange(0, 1_000_000 + 10_000, 10_000)
    counts, edges = np.histogram(isi_us, bins=bins_us)
    if counts.size > 0:
        isi_peak_bin = int(np.argmax(counts))
        isi_peak_center_us = (edges[isi_peak_bin] + edges[isi_peak_bin + 1]) / 2.0
        isi_peak_count = int(counts[isi_peak_bin])
    else:
        isi_peak_center_us = np.nan
        isi_peak_count = 0

    # ---- Figure ----
    fig = plt.figure(figsize=(8, 6))
    gs = fig.add_gridspec(nrows=2, ncols=1, height_ratios=[3, 2], hspace=0.35)

    # Top: Waveform
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.plot(t_us, mean_wv_uV, c='black', linestyle='dashed', linewidth=2, label='Mean waveform')
    ax1.plot(t_us[:mean_temp_uV.shape[-1]], mean_temp_uV, linewidth=1, label='Template')
    ax1.scatter(t_us[peak_idx], mean_wv_uV[peak_idx], c='blue', zorder=5)
    ax1.scatter(t_us[trough_idx], mean_wv_uV[trough_idx], c='red', zorder=5)

    y_min = min(mean_wv_uV.min(), mean_temp_uV.min())
    y_max = max(mean_wv_uV.max(), mean_temp_uV.max())
    ax1.vlines([t_us[peak_idx], t_us[trough_idx]], ymin=y_min, ymax=y_max,
               linestyles='dotted', colors=['blue', 'red'], alpha=0.6)
    y_mid = (mean_wv_uV[peak_idx] + mean_wv_uV[trough_idx]) / 2.0
    ax1.hlines(y=y_mid, xmin=t_us[peak_idx], xmax=t_us[trough_idx],
               colors='purple', alpha=0.85)
    ax1.text((t_us[peak_idx] + t_us[trough_idx]) / 2.0, y_mid,
             f"Δt = {dt_us:.1f} µs", ha='center', va='bottom', fontsize=9)

    title = f"{session_name} | Cluster {cluster_id} | Best ch {best_chan} | FR {fr_hz:.2f} Hz | n={n_spikes}"
    ax1.set_title(title)
    ax1.set_xlabel("Time (µs)")
    ax1.set_ylabel("Voltage (µV)")

    # Bottom: ISI
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(isi_us, bins=bins_us)
    ax2.set_title("ISI Histogram (10 ms bins, up to 1 s)")
    ax2.set_xlabel("ISI (µs)")
    ax2.set_ylabel("Count")
    if np.isfinite(isi_peak_center_us):
        ax2.axvline(isi_peak_center_us, linestyle='--', color='k')
        ax2.text(isi_peak_center_us, ax2.get_ylim()[1]*0.9,
                 f"Peak ≈ {int(isi_peak_center_us)} µs\nCount={isi_peak_count}",
                 ha='center', va='top', fontsize=8)

    fig.savefig(save_path_png, dpi=150, bbox_inches='tight')
    plt.close(fig)

    return {
        "fr_hz": fr_hz,
        "n_spikes": n_spikes,
        "isi_peak_center_us": isi_peak_center_us,
        "isi_peak_count": isi_peak_count,
        "peak_time_us": wf["peak_time_us"],
        "trough_time_us": wf["trough_time_us"],
        "peak_to_trough_us": wf["peak_to_trough_us"],
        "peak_uV": wf["peak_uV"],
        "trough_uV": wf["trough_uV"]
    }

# ---------------------- per-cluster processing ----------------------

def process_cluster(cluster_id, session_name, results_dir, fs_default=30000):
    results_dir = Path(results_dir)

    figures_dir = results_dir / "figures" / session_name
    figures_dir.mkdir(parents=True, exist_ok=True)

    fs = fs_default
    try:
        ops = load_ops(results_dir / 'ops.npy')
        fs = float(ops.get('fs', fs_default))
    except Exception:
        pass

    mean_wv, spike_subset = mean_waveform(cluster_id, results_dir, n_spikes=100, best=True)
    mean_temp = cluster_templates(cluster_id, results_dir, mean=True, best=True, spike_subset=spike_subset)
    best_chan = get_best_channel(results_dir, cluster_id) + 1

    t_us = (np.arange(mean_wv.shape[-1]) / fs) * 1e6

    spikes = np.load(results_dir / 'spike_times.npy')
    labels = np.load(results_dir / 'spike_clusters.npy')
    cluster_spikes = np.sort(spikes[labels == cluster_id])

    fig_path = figures_dir / f"{session_name}_cluster{cluster_id}.png"
    fig_metrics = waveform_isi_fig(
        t_us=t_us,
        mean_wv_uV=mean_wv,
        mean_temp_uV=mean_temp,
        fs=fs,
        cluster_id=cluster_id,
        session_name=session_name,
        best_chan=best_chan,
        cluster_spikes=cluster_spikes,
        all_spikes=spikes,
        save_path_png=fig_path
    )

    ts_dir = results_dir / "timestamps"
    ts_dir.mkdir(exist_ok=True)
    sio.savemat(ts_dir / f"{cluster_id}.mat",
                {f"cluster_{cluster_id}_timestamp": cluster_spikes[:, None]})

    return {
        "session": session_name,
        "cluster_id": int(cluster_id),
        "best_channel": int(best_chan),
        "peak_time_us": fig_metrics["peak_time_us"],
        "trough_time_us": fig_metrics["trough_time_us"],
        "peak_to_trough_us": fig_metrics["peak_to_trough_us"],
        "peak_uV": fig_metrics["peak_uV"],
        "trough_uV": fig_metrics["trough_uV"],
        "isi_peak_center_us": float(fig_metrics["isi_peak_center_us"]) if np.isfinite(fig_metrics["isi_peak_center_us"]) else np.nan,
        "isi_peak_count": int(fig_metrics["isi_peak_count"]),
        "avg_firing_rate_Hz": float(fig_metrics["fr_hz"]) if np.isfinite(fig_metrics["fr_hz"]) else np.nan,
        "n_spikes": int(fig_metrics["n_spikes"]),
    }

# ---------------------- NEW TOP-LEVEL FUNCTION ----------------------

def kilosort_postprocess_all_good(results_dir, session_name):
    results_dir = Path(results_dir)

    label_path = results_dir / "cluster_KSLabel.tsv"
    if not label_path.exists():
        raise FileNotFoundError("Missing cluster_KSLabel.tsv")

    kslabels = pd.read_csv(label_path, sep="\t")
    good_clusters = kslabels.loc[kslabels["KSLabel"] == "good", "cluster_id"].astype(int).tolist()

    print(f"[INFO] Found {len(good_clusters)} good units.")

    fig_dir = results_dir / "figures" / session_name
    if fig_dir.exists():
        rmtree(fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)

    ts_dir = results_dir / "timestamps"
    if ts_dir.exists():
        rmtree(ts_dir)
    ts_dir.mkdir(parents=True, exist_ok=True)

    session_rows = []
    best_rows = []

    for cid in good_clusters:
        print(f"Processing GOOD cluster {cid}")
        try:
            row = process_cluster(cid, session_name, results_dir)
            session_rows.append(row)
            best_rows.append({
                "session": session_name,
                "cluster_id": row["cluster_id"],
                "best_channel": row["best_channel"]
            })
        except Exception as e:
            print(f"[ERROR] cluster {cid} → {e}")

    # tidy excel
    tidy_path = results_dir / f"{session_name}.xlsx"
    if tidy_path.exists():
        tidy_path.unlink()

    df = pd.DataFrame(session_rows)
    preferred = [
        "session","cluster_id","best_channel",
        "peak_to_trough_us","peak_time_us","trough_time_us",
        "peak_uV","trough_uV",
        "avg_firing_rate_Hz","n_spikes",
        "isi_peak_center_us","isi_peak_count",
    ]
    df = df[[c for c in preferred if c in df.columns]] 
    df.to_excel(tidy_path, index=False)
    print(f"[OK] Wrote tidy Excel → {tidy_path}")

    # best_channels
    bc_path = results_dir / "best_channels.xlsx"
    if bc_path.exists():
        bc_path.unlink()

    pd.DataFrame(best_rows).sort_values("cluster_id").to_excel(bc_path, index=False)
    print(f"[OK] Wrote best channels → {bc_path}")

    print("Done.")


In [3]:
kilosort_postprocess_all_good(r'D:\Data\Chronic Recordings m5\Rearranged Channles\m5s26jan19\2024-01-19_12-53-25\kilosort4', 'm5s26chronic')

[INFO] Found 51 good units.
Processing GOOD cluster 1
Processing GOOD cluster 21
Processing GOOD cluster 30
Processing GOOD cluster 43
Processing GOOD cluster 52
Processing GOOD cluster 65
Processing GOOD cluster 67
Processing GOOD cluster 76
Processing GOOD cluster 83
Processing GOOD cluster 84
Processing GOOD cluster 91
Processing GOOD cluster 94
Processing GOOD cluster 166
Processing GOOD cluster 180
Processing GOOD cluster 184
Processing GOOD cluster 193
Processing GOOD cluster 222
Processing GOOD cluster 236
Processing GOOD cluster 388
Processing GOOD cluster 390
Processing GOOD cluster 398
Processing GOOD cluster 399
Processing GOOD cluster 402
Processing GOOD cluster 403
Processing GOOD cluster 414
Processing GOOD cluster 416
Processing GOOD cluster 417
Processing GOOD cluster 420
Processing GOOD cluster 425
Processing GOOD cluster 427
Processing GOOD cluster 428
Processing GOOD cluster 443
Processing GOOD cluster 445
Processing GOOD cluster 448
Processing GOOD cluster 451
Proce